# Loss Functions in Deep Learning

---

## 1. Introduction

A **loss function** (also called an objective function or criterion) is a mathematical function that measures how well a machine learning model's predictions match the actual target values. It quantifies the *error* or *discrepancy* between predicted outputs and ground truth labels.

> **Why does it matter?** You can't improve what you can't measure. The loss function provides the signal that guides the optimizer (e.g., gradient descent) to update the model's weights during training.

![Different Loss Functions](./images/14_1.png)

---

## 2. Loss Function vs. Cost Function

These two terms are often used interchangeably, but there is a subtle distinction:

| Term | Scope | Formula |
|------|-------|---------|
| **Loss Function** `L` | A **single** training example | `L(ŷ⁽ⁱ⁾, y⁽ⁱ⁾)` |
| **Cost Function** `J` | The **entire** training dataset | `J = (1/m) Σ L(ŷ⁽ⁱ⁾, y⁽ⁱ⁾)` |

- `m` = number of training examples
- `y⁽ⁱ⁾` = true label for example *i*
- `ŷ⁽ⁱ⁾` = predicted label for example *i*

In practice, the **cost function** is what we minimize during training via backpropagation.

---

## 3. Regression Loss Functions

Used when the target variable is **continuous** (e.g., predicting house prices, temperatures).

---

### 3.1 Mean Squared Error (MSE) — L2 Loss

**Loss Function:**

$$L(y, \hat{y}) = (y - \hat{y})^2$$

**Cost Function:**

$$J = \frac{1}{m} \sum_{i=1}^{m} (y^{(i)} - \hat{y}^{(i)})^2$$

**How it works:** Penalizes large errors more heavily due to squaring. The error is always non-negative, and the function is convex with a single global minimum.

| ✅ Advantages | ❌ Disadvantages |
|---|---|
| Differentiable everywhere | Error unit is squared (harder to interpret) |
| Single global minimum (convex) | Very sensitive to **outliers** |
| Smooth gradients, stable training | Large errors get disproportionately amplified |

**When to use:**
- Target is continuous and roughly normally distributed
- Outliers are not a concern or have been removed
- You want a smooth, well-behaved optimization landscape

---

### 3.2 Mean Absolute Error (MAE) — L1 Loss

**Loss Function:**

$$L(y, \hat{y}) = |y - \hat{y}|$$

**Cost Function:**

$$J = \frac{1}{m} \sum_{i=1}^{m} |y^{(i)} - \hat{y}^{(i)}|$$

**How it works:** Measures the average absolute difference between predictions and actual values. Treats all errors equally regardless of their magnitude.

| ✅ Advantages | ❌ Disadvantages |
|---|---|
| Same unit as the target (easy to interpret) | **Not differentiable** at zero |
| **Robust to outliers** | Constant gradient can slow convergence near minimum |
| Good when median prediction is preferred | Requires subgradient methods for optimization |

**When to use:**
- Your dataset contains significant outliers
- You want an interpretable, same-unit error metric
- Median-based predictions are preferred over mean-based

---

### 3.3 Huber Loss (Smooth MAE)

**Loss Function:**

$$L_{\delta}(y, \hat{y}) = \begin{cases} \frac{1}{2}(y - \hat{y})^2 & \text{if } |y - \hat{y}| \leq \delta \\ \delta \cdot |y - \hat{y}| - \frac{1}{2}\delta^2 & \text{otherwise} \end{cases}$$

**How it works:** A hybrid of MSE and MAE. Behaves like **MSE for small errors** (smooth, differentiable) and like **MAE for large errors** (robust to outliers). The hyperparameter `δ` (delta) controls the transition point between the two behaviors.

| ✅ Advantages | ❌ Disadvantages |
|---|---|
| Differentiable everywhere (unlike MAE) | Requires tuning the hyperparameter `δ` |
| Robust to outliers (unlike MSE) | Slightly more complex to implement |
| Combines the best of MSE and MAE | Less commonly available out-of-the-box |

**When to use:**
- You want outlier robustness but still need differentiability
- Commonly used in **reinforcement learning** (e.g., DQN)
- A balanced alternative when neither MSE nor MAE alone is ideal

---

## 4. Classification Loss Functions

Used when the target variable is **categorical** (e.g., classifying images, detecting spam).

---

### 4.1 Binary Cross-Entropy (Log Loss)

**Loss Function:**

$$L(y, \hat{y}) = -\left[ y \log(\hat{y}) + (1 - y) \log(1 - \hat{y}) \right]$$

**Cost Function:**

$$J = -\frac{1}{m} \sum_{i=1}^{m} \left[ y^{(i)} \log(\hat{y}^{(i)}) + (1 - y^{(i)}) \log(1 - \hat{y}^{(i)}) \right]$$

**How it works:** Measures the difference between two probability distributions — the true binary label (0 or 1) and the model's predicted probability. A perfect prediction yields zero loss; a confident wrong prediction yields very high loss.

| ✅ Advantages | ❌ Disadvantages |
|---|---|
| Strong probabilistic interpretation | Sensitive to **class imbalance** |
| Works well with sigmoid activation | Requires predictions in range (0, 1) |
| Differentiable with smooth gradients | Heavily penalizes confident wrong predictions |

**When to use:**
- Binary classification problems (spam/not spam, cat/dog)
- Output layer uses **sigmoid activation**
- Multi-label classification (each label treated independently)

---

### 4.2 Categorical Cross-Entropy

**Loss Function (single example):**

$$L(y, \hat{y}) = -\sum_{c=1}^{C} y_c \log(\hat{y}_c)$$

**Cost Function:**

$$J = -\frac{1}{m} \sum_{i=1}^{m} \sum_{c=1}^{C} y_c^{(i)} \log(\hat{y}_c^{(i)})$$

Where `C` is the number of classes, and `y` is a **one-hot encoded** vector.

**How it works:** A generalization of binary cross-entropy to multiple classes. The true label is represented as a one-hot vector, and the model outputs a probability distribution over all classes via softmax.

| ✅ Advantages | ❌ Disadvantages |
|---|---|
| Standard loss for multi-class problems | Requires **one-hot encoded** labels |
| Works directly with softmax outputs | Memory-intensive for very large label spaces |
| Strong probabilistic foundation | Sensitive to class imbalance |

**When to use:**
- Multi-class classification with **mutually exclusive** classes
- Output layer uses **softmax activation**
- Labels are already one-hot encoded

---

### 4.3 Sparse Categorical Cross-Entropy

**Loss Function:**

$$L(y, \hat{y}) = -\log(\hat{y}_{y})$$

Where $y$ is the **integer class index**, and $\hat{y}_{y}$ is the predicted probability for the correct class.

**How it works:** Mathematically identical to Categorical Cross-Entropy, but accepts **integer class labels** directly instead of one-hot vectors. This avoids the memory and preprocessing overhead of one-hot encoding, especially when the number of classes is large.

| ✅ Advantages | ❌ Disadvantages |
|---|---|
| Memory-efficient — no one-hot encoding needed | Cannot directly express label uncertainty |
| Convenient for large label spaces (e.g., NLP) | Less flexible for soft/mixed labels |
| Equivalent performance to Categorical CE | Requires integer-indexed labels |

**When to use:**
- Multi-class classification where labels are **integers** (not one-hot)
- Large number of classes (e.g., NLP tasks with large vocabularies)
- When you want to skip label preprocessing

---

## 5. Summary Comparison

| Loss Function | Task | Robust to Outliers | Differentiable | Label Format |
|---|---|---|---|---|
| MSE (L2) | Regression | ❌ | ✅ | Continuous |
| MAE (L1) | Regression | ✅ | ❌ (at 0) | Continuous |
| Huber | Regression | ✅ | ✅ | Continuous |
| Binary Cross-Entropy | Binary Classification | — | ✅ | 0 or 1 |
| Categorical Cross-Entropy | Multi-class Classification | — | ✅ | One-hot |
| Sparse Categorical Cross-Entropy | Multi-class Classification | — | ✅ | Integer index |

---

## 6. Quick Selection Guide

```
Is it a regression problem?
├── Are there outliers?
│   ├── Yes → Huber Loss or MAE
│   └── No  → MSE
└── Is it a classification problem?
    ├── 2 classes?          → Binary Cross-Entropy
    ├── Multiple classes?
    │   ├── One-hot labels? → Categorical Cross-Entropy
    │   └── Integer labels? → Sparse Categorical Cross-Entropy
```